# ISA 591 · Module 6 · Class 3
## Predictive Logistic Regression: Regularization, Convergence, and Evaluation

**80-minute interactive worksheet · `scikit-learn`**

**Business question:** Which bank customers are likely to accept a personal-loan offer? We aim to make useful predictions for **new** customers, not interpret p-values.

**By the end:** prepare data, split into train and test, fit `LogisticRegression`, examine `predict_proba`, compare classification metrics, understand L1 versus L2 regularization and `C`, and perform a simple `GridSearchCV` on training data.

**Class plan:** Prepare/split (15 min) · Fit/evaluate (20 min) · L1/L2 and convergence (20 min) · Grid search and ROC (20 min) · Wrap-up (5 min).

### 1. Load and inspect the Universal Bank dataset (0–10 min)
The target `Personal Loan` is 1 if a customer accepted a personal loan. This is a **different business problem** from the default-rate example in Class 1. The course URL requires internet access; a simulated practice CSV is included in the ZIP as an alternative.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Original course dataset
df = pd.read_csv('https://raw.githubusercontent.com/martinwg/ISA591/refs/heads/main/data/UniversalBank.csv')
# Offline alternative (uncomment when using the ZIP package):
# df = pd.read_csv('ISA591_Module6_UniversalBank_Simulated_Practice.csv')

display(df.head())
print('Rows:', len(df))
print('Personal-loan acceptance rate:', round(df['Personal Loan'].mean(), 3))

,ID,Personal Loan,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,Securities Account,CD Account,Online,CreditCard
0,1,0,25,1,49,91107,4,1.6,1,0,1,0,0,0
1,2,0,45,19,34,90089,3,1.5,1,0,1,0,0,0
2,3,0,39,15,11,94720,1,1.0,1,0,0,0,0,0
3,4,0,35,9,100,94112,1,2.7,2,0,0,0,0,0
4,5,0,35,8,45,91330,4,1.0,2,0,0,0,0,1


Rows: 5000
Personal-loan acceptance rate: 0.096


In [ ]:
## y = {1: 'accepted a personal loan offer', 0: 'did not accept'}
#### interested in understanding the P[accept]

### 2. Prepare predictors (10–15 min)
We drop `ID` and `ZIP Code` (identifiers/location codes not used in this exercise). Treat `Education` as categorical rather than pretending category labels are numerical amounts. Use `pd.get_dummies(..., drop_first=True)` to avoid a redundant dummy column.

In [3]:
## 1) drop ID and ZIP Code
df.drop(['ID', 'ZIP Code'], axis = 1, inplace = True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Personal Loan       5000 non-null   int64  
 1   Age                 5000 non-null   int64  
 2   Experience          5000 non-null   int64  
 3   Income              5000 non-null   int64  
 4   Family              5000 non-null   int64  
 5   CCAvg               5000 non-null   float64
 6   Education           5000 non-null   int64  
 7   Mortgage            5000 non-null   int64  
 8   Securities Account  5000 non-null   int64  
 9   CD Account          5000 non-null   int64  
 10  Online              5000 non-null   int64  
 11  CreditCard          5000 non-null   int64  
dtypes: float64(1), int64(11)
memory usage: 468.9 KB


In [4]:
## 2) Encode as Categorical
df.Education = df.Education.astype('O')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Personal Loan       5000 non-null   int64  
 1   Age                 5000 non-null   int64  
 2   Experience          5000 non-null   int64  
 3   Income              5000 non-null   int64  
 4   Family              5000 non-null   int64  
 5   CCAvg               5000 non-null   float64
 6   Education           5000 non-null   object 
 7   Mortgage            5000 non-null   int64  
 8   Securities Account  5000 non-null   int64  
 9   CD Account          5000 non-null   int64  
 10  Online              5000 non-null   int64  
 11  CreditCard          5000 non-null   int64  
dtypes: float64(1), int64(10), object(1)
memory usage: 468.9+ KB


In [5]:
## 3) Get dummy-encode vars for Education
df = pd.get_dummies(df, drop_first = True, dtype = int)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Personal Loan       5000 non-null   int64  
 1   Age                 5000 non-null   int64  
 2   Experience          5000 non-null   int64  
 3   Income              5000 non-null   int64  
 4   Family              5000 non-null   int64  
 5   CCAvg               5000 non-null   float64
 6   Mortgage            5000 non-null   int64  
 7   Securities Account  5000 non-null   int64  
 8   CD Account          5000 non-null   int64  
 9   Online              5000 non-null   int64  
 10  CreditCard          5000 non-null   int64  
 11  Education_2         5000 non-null   int32  
 12  Education_3         5000 non-null   int32  
dtypes: float64(1), int32(2), int64(10)
memory usage: 468.9 KB


In [6]:
df.head()

,Personal Loan,Age,Experience,Income,Family,CCAvg,Mortgage,Securities Account,CD Account,Online,CreditCard,Education_2,Education_3
0,0,25,1,49,4,1.6,0,1,0,0,0,0,0
1,0,45,19,34,3,1.5,0,1,0,0,0,0,0
2,0,39,15,11,1,1.0,0,0,0,0,0,0,0
3,0,35,9,100,1,2.7,0,0,0,0,0,1,0
4,0,35,8,45,4,1.0,0,0,0,0,1,1,0


**TODO:** Why would including an identifier such as `ID` be unhelpful? Why use `drop_first=True` when creating dummies?

In [7]:
## For linear models we need non-collinearity (no perfect collinearity) - not the case for non-linear models
#### Education = [1,2,3]
######## Education_2 = {0,1}, Education_3 = {0,1}

In [ ]:
## statmodels is updated to work with bool values
sm.__version__

# !pip install --upgrade statsmodels

In [9]:
## DESCRIPTIVE
import statsmodels.api as sm 

## create instance and fit
lr = sm.Logit(df['Personal Loan'], sm.add_constant(df.drop('Personal Loan', axis = 1))).fit()

## summary
print(lr.summary())

Optimization terminated successfully.
         Current function value: 0.117230
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:          Personal Loan   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4987
Method:                           MLE   Df Model:                           12
Date:                Wed, 18 Nov 2026   Pseudo R-squ.:                  0.6293
Time:                        10:44:24   Log-Likelihood:                -586.15
converged:                       True   LL-Null:                       -1581.0
Covariance Type:            nonrobust   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                -12.3105      1.818     -6.773      0.000     -15.873      -8.748
Age  

In [ ]:
## As age increases does that increase or decrease the likelihood to accept this personal loan offer?
#### As age increases, the "likelihood" to accept the offer decreases if everything else stays constant.
#### However, this variable is NOT statistically signficant

## "Log(odds)  -- Probability --- Likelihood" all move in the same direction. Signs are the same

## Having a mortgage with the company increases the probability to accept the loan of offer by 0.0007 holding ovc? True or False
### FALSE - log(odds) and probability might be close but they are not exactly 
### Having a mortgage increases the LOG(ODDS) of accepting the offer by 0.0007 holding other vars constant (not significant)


### 3. Fit predictive logistic regression (15–27 min)
`scikit-learn` focuses on predictions and regularized coefficient estimation. Its `LogisticRegression` uses L2 regularization by default. We use `max_iter` to allow the optimizer more iterations when needed.

In [19]:
## LOSS FUNCTION OF LOGISTIC REGRESSION
#### log-loss 
#### a loss is a calculation of the error of a model, e.g., in LR the squared loss: (y - yhat)^2  (100 - 20)^2
#### for classification y = 1 (accepted), p = 0.95 
y = 1          ## accepted
p = 0.001      ## prob given by the model

-(y*np.log(p) + (1-y)*np.log(1-p))

6.907755278982137

In [ ]:
## Cost is the sum of ALL the loss values that we get
cost = 6.91

-9.210340371976182

In [ ]:
## How is a predictive MODEL going to discourage using variables that ARE NOT SIGNIFICANT?
#### We add penalties to the cost function

#### 1) L2 norm
##### cost + penalty
##### cost + [slope1^2 + slope2^2 ....] 
##### this forces the slopes to be as close to zero as possible (focus on the unimportant slopes)
##### We can control the EMPHASIS on the penalty by adding a weight (we call C) - lower C more emphasis on the penalty
##### cost + C[slope1^2 + slope2^2 ....] 

In [ ]:
## PREDICTIVE LOGISTIC REGRESSION
#### WE use sklearn (DOES NOT output p-values, stats)

## 1) split 
X = df.drop('Personal Loan', axis = 1)
y = df['Personal Loan']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=591, stratify=y)

from sklearn.linear_model import LogisticRegression

## 2) instance
#### this is NOT THE same as a Logistic Regression fit with statsmodels 
#### statmodels LR uses the: COST in the gradient descent
#### sklearn uses the:       COST + C*L2 in the gradient descent [to discourage statistically insignificant vars]
lr = LogisticRegression(penalty = "l2", max_iter = 1000)
### what happens if max_iter is TOO large and no convergence, change solver = 'liblinear' 
## lr = LogisticRegression(penalty = "l2", solver = 'liblinear')

## 3) fit
lr.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

Model output includes predicted **classes** and predicted **probabilities**. They serve different purposes.

### 4. Evaluate the model (27–40 min)
The default classification cutoff is 0.5. Check **accuracy, recall, precision**, and the confusion matrix. If fewer than half the customers accept the offer, accuracy alone can be misleading.

**TODO:** In a marketing campaign, when might recall matter more than precision? What would happen if you contacted everyone (always predicted class 1)?

### 5. Regularization: L2 versus L1 (40–55 min)
**L2** tends to shrink coefficients toward zero; **L1** may set some exactly to zero. In `sklearn`, **smaller `C` means stronger regularization**. For simplicity we follow the direct fitting approach used in class. Coefficient magnitudes also depend on predictor units; a full modeling workflow should scale continuous inputs inside cross-validation.

Compare how many coefficients are exactly zero. Do not interpret absolute magnitudes across predictors measured in very different units.

**TODO:** Change `C=0.1` to `C=0.01` and rerun the two previous cells. Does stronger regularization make more L1 coefficients zero?

### 6. Convergence and GridSearchCV (55–68 min)
The model is fitted iteratively. If you see a *convergence warning*, investigate feature scales, `max_iter`, and the solver. Do not automatically assume more iterations solve every problem.

`GridSearchCV` checks combinations of `C` and penalty using **cross-validation on the training data**. For binary classification, `roc_auc` measures ranking performance independent of a particular cutoff.

### 7. Final test evaluation (68–75 min)
Use the fitted `grid.best_estimator_` on the held-out test set. The test set is **not used to select `C`, the penalty, or the decision threshold**.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, log_loss

best_lr = grid.best_estimator_
prob_test = best_lr.predict_proba(X_test)[:, 1]

print('Test ROC-AUC:', round(roc_auc_score(y_test, prob_test), 3))
print('Test Log Loss:', round(log_loss(y_test, prob_test), 3))

fpr, tpr, thresholds = roc_curve(y_test, prob_test)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label='ROC-AUC = ' + str(round(roc_auc_score(y_test, prob_test), 3)))
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve — Held-Out Test Data')
plt.legend()
plt.show()

### 8. Optional cutoff demonstration
These are **illustrative preset thresholds**, not cutoffs optimized using the test set. Lowering a classification threshold generally increases the number of customers predicted to accept an offer. To *choose* a threshold, use training/validation data and a stated business objective.

In [ ]:
# TODO: Try 0.30 and 0.70; observe precision/recall tradeoffs.
threshold = 0.30
pred_at_threshold = (prob_test >= threshold).astype(int)

print('Classification cutoff:', threshold)
print(confusion_matrix(y_test, pred_at_threshold))
print(classification_report(y_test, pred_at_threshold, zero_division=0))

### Exit ticket (75–80 min)
1. What does `predict_proba()[:, 1]` give us?
2. When `C` decreases, does regularization become stronger or weaker?
3. Why do we perform `GridSearchCV` on the **training set** rather than the test set?
4. How would this exercise change if the goal were explanatory *p-values* instead of predictions?

**Optional extension:** Repeat the grid using `scoring='recall'`. Does the chosen model change? Remember that the default recall scorer uses the classifier's 0.5 threshold.